In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Read dataset
df = pd.read_csv(path + "/Q1_data.csv")

df.columns = df.columns.str.strip()
df.columns = df.columns.str.lower()

In [ ]:
# Task 2: Inspect first few rows
print(df.head())

In [ ]:
# Task 3: Display dataset info
print(df.info())

In [ ]:
# Task 4: Statistical description
print(df.describe())

In [ ]:
# Task 5: Plot target distribution
plt.figure(figsize=(8,5))
sns.histplot(df['delivery_time'], bins=30, kde=True, color='blue')
plt.title("Distribution of Delivery Time")
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Task 1: Drop 'Order_ID'
if 'order_id' in df.columns:
    df = df.drop('order_id', axis=1)

In [ ]:
# Task 2: Handle missing values
print(df.isnull().sum())   # Inspect missing values
df = df.fillna(df.median(numeric_only=True))  # Fill numeric with median
df = df.fillna(df.mode().iloc[0])             # Fill categorical with mode

In [ ]:
# Task 3: Remove duplicates
df = df.drop_duplicates()

In [ ]:
# Task 4: Encode categorical variables (One Hot Encoding)
df = pd.get_dummies(df, drop_first=True)

In [ ]:
# Task 5: Feature scaling
scaler = StandardScaler()
features = df.drop(columns=['delivery_time'])
scaled_features = scaler.fit_transform(features)
X = pd.DataFrame(scaled_features, columns=features.columns)

In [ ]:
# Task 6: Check target imbalance
sns.histplot(df['delivery_time'], bins=30, kde=True)
plt.title("Target Distribution Check")
plt.show()

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Task 1: Split dataset
y = df['delivery_time']

In [ ]:
# Task 2,3,4,5: KFold + RandomForest + MAE
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

print("Average MAE across folds:", np.mean(mae_scores))

In [ ]:
# Task 1: Feature importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10,6))
sns.barplot(x=importances[indices], y=X.columns[indices], palette="viridis")
plt.title("Feature Importance")
plt.show()

In [ ]:
# Task 2: Predicted delivery time histogram
plt.figure(figsize=(8,5))
sns.histplot(y_pred, bins=30, kde=True, color='green')
plt.title("Predicted Delivery Time Distribution")
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
import numpy as np

# Features and target
X = df.drop(columns=['delivery_time'])
y = df['delivery_time']

# KFold setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Model 1: RandomForest
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_preds = rf_model.predict(X_test)

    # Model 2: CatBoost
    cb_model = CatBoostRegressor(iterations=200, depth=6, learning_rate=0.1,
                                 loss_function='MAE', verbose=0, random_state=42)
    cb_model.fit(X_train, y_train)
    cb_preds = cb_model.predict(X_test)

    # Average predictions
    avg_preds = (rf_preds + cb_preds) / 2

    # MAE on averaged predictions
    mae = mean_absolute_error(y_test, avg_preds)
    mae_scores.append(mae)

print("Average MAE across folds (Ensemble):", np.mean(mae_scores))